In [1]:
# Configure path.
import sys; import os
sys.path.append(os.path.abspath('..'))

%matplotlib inline
import numpy as np
import importlib
import matplotlib.pyplot as plt
plt.style.use('dissertation.mplstyle')

import python.forces as forces
import python.integrators as integrators
import python.random_matrix as random_matrix 
import python.simulate as simulate
import python.densities as densities
import python.solvers as solvers

importlib.reload(forces); importlib.reload(integrators); importlib.reload(random_matrix)
importlib.reload(simulate); importlib.reload(densities); importlib.reload(solvers);

---

Some figures for the summary notes.
- Empirical density for quadratic (semicircle) and quartic potentials for: tamed, implicit, imla, maimla schemes. In all cases use $N = 70$ and a reasonable choice of $dt$ for each scheme. With $M = 100$. Run until $T = 10$ and take snapshots past $T = 3$.
- Wasserstein distance for the same trajectories.
- See `04_mala.ipynb` for the crossing rejects in MALA.

In [ ]:
importlib.reload(simulate);

N = 120; beta = 2.0; T = 10.0; M = 500;
pot_name = "quadratic"; num_bins = 100;

# Exact CDF used for the Wasserstein distances.
grid = np.linspace(-2, 2, 1000)
F_sc = densities.exact_semicircle_cdf(grid)
F_grid, F_pdf = densities.get_density("quadratic")

dts = {
    "tamed": 1/N**2,
    "implicit": 0.05,
    "imla": 0.05,
    "maimla": 1/N
}

# tamed_pipe = simulate.get_pipeline("tamed", dt = dts["tamed"], noise_scale = np.sqrt(2*dts["tamed"]/(beta*N)), potential_type = pot_name)
implicit_pipe = simulate.get_pipeline("implicit", dt = dts["implicit"], noise_scale = np.sqrt(2*dts["implicit"]/(beta*N)), potential_type = pot_name)
imla_pipe = simulate.get_pipeline("imla", dt = dts["imla"], noise_scale = np.sqrt(2*dts["imla"]/(beta*N)), potential_type = pot_name, beta = beta, metropolise = False)
maimla_pipe = simulate.get_pipeline("imla", dt = dts["imla"], noise_scale = np.sqrt(2*dts["maimla"]/(beta*N)), potential_type = pot_name, beta = beta, metropolise = True)

pipes = [implicit_pipe, imla_pipe, maimla_pipe]
methods = ["implicit", "imla", "maimla"]
distance_infos = []

# Same for all methods.
init = np.random.normal(0, 1, (M, N))

for pipe, method in zip(pipes, methods):
    print(f"Running {method} algorithm.")
    dt = dts[method]
    total_steps = int(T/dt); burn_in = int(3.0/dt)

    traj = simulate.simulate_dbm(init, total_steps, pipe)
    particles, distance_info = simulate.collect_snapshots_distance(traj, grid, F_sc, total_steps, dt = dts[method], burn_in = burn_in)

    plot_hist(particles, F_grid, F_pdf, method_name = method)
    distance_infos.append(distance_info)

fig, ax = plt.subplots()
for info, method in zip(distance_infos, methods):
    ax.plot(info["steps"], info["distances"], label = method)

ax.set_yscale("log")
ax.legend(loc = "upper right")

plt.show()